In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l5.1 Initialization

> 🎯 **Goal:** Start every weight from the right kind of random so the very first training loss is finite, sane, and close to the no-knowledge baseline.

**Why this matters.** A neural network is a giant pile of numbers (its weights) that get nudged during training. Before training even begins, those numbers have to start somewhere. If they start in a bad place, training never recovers: the loss (the single number that measures how wrong the model is) can blow up to NaN (Not a Number, the result of a computation like infinity minus infinity or zero divided by zero) on the first step, and there is no coming back from NaN. Good initialization is the launchpad that every later lesson in this unit stands on.

**The intuition.** Think of initialization as choosing where on a mountain to drop a hiker before they start walking downhill toward the valley (the low loss we want). Drop them on a vertical cliff face and they tumble out of control: that is the loss exploding to NaN. Drop them deep in a featureless flat where every direction looks the same and they barely move: that is signal failing to flow. The GPT recipe drops the hiker on a gentle, walkable slope, near the natural starting altitude a fresh model should have.

**The mechanics.** The standard GPT initialization fills the weights from a normal (bell-curve) distribution with a small standard deviation, std = 0.02. Standard deviation is just the typical size of the random values: 0.02 keeps them small, so signals neither explode nor vanish as they pass through the layers. Why does this land near `ln(vocab)`? A brand-new model knows nothing, so for each next character it should spread its bet roughly evenly across all `vocab_size` possible characters. The loss of that even-bet strategy is exactly `ln(vocab_size)` in nats (the natural-log unit of information). So a correctly initialized model reports a first loss right around `ln(vocab)`, confirming it starts unbiased, knowing nothing yet, but ready to learn.

In [ ]:
import math
from pocketlm import PocketLM, PocketLMConfig, init_weights

torch.manual_seed(0)
cfg = PocketLMConfig(vocab_size=20, block_size=16, n_layer=2, n_head=2, n_embd=32)
noise = torch.randint(0, 20, (200,))
xb = noise[:16].unsqueeze(0)
yb = noise[1:17].unsqueeze(0)
model = init_weights(PocketLM(cfg), std=0.02)
_, loss = model(xb, yb)
print("init loss:", round(loss.item(), 3), "  ln(vocab):", round(math.log(20), 3))

**What just happened.** The printout shows two numbers: the model's first loss and `ln(vocab)`. They should be close, and both should be small and finite. That tells you the weights landed on a walkable slope. The model has not learned anything yet (it cannot, it has taken zero training steps) but it is poised exactly at the unbiased baseline, ready to descend.

⚠️ **Common pitfalls**
- Using too large a std (say 1.0 instead of 0.02): activations and gradients blow up, and the first loss is huge or NaN with no recovery.
- Using too small a std (say 1e-6): everything starts near zero, signals barely propagate, and the model trains painfully slowly or not at all.
- Forgetting to seed the random generator (`torch.manual_seed`): your init differs every run, so a flaky result looks like a code bug when it is just unlucky randomness.

🏋️ **Try it yourself.** Re-run the cell with `std=1.0` and then `std=0.5`, and watch the first loss climb far above `ln(vocab)`. Then try `std=0.001` and notice how the loss sits close to baseline but the model will be sluggish to train. The sweet spot of 0.02 is no accident.

In [ ]:
import math
assert torch.isfinite(loss)
assert loss.item() < 2 * math.log(20)